## **Task 1**

In [ ]:
import numpy as np

In [15]:
def quantize_int8(values):
    values = np.array(values, dtype=np.float32) # ek array

    qmin = -128
    qmax = 127

    xmin = np.min(values)
    xmax = np.max(values)

    scale = (xmax - xmin) / (qmax - qmin)

    zero_point = qmin - (xmin / scale)
    zero_point = np.round(zero_point)
    zero_point = np.clip(zero_point, qmin, qmax)

    quantized = np.round(values / scale + zero_point)
    quantized = np.clip(quantized, qmin, qmax)

    return quantized.astype(np.int8), scale, int(zero_point)

In [18]:
data = [-1.0, -0.5, 0.0, 0.5, 1.0]

quantized_data, scale, zero_point = quantize_int8(data)

print("Original values:", data)
print("Quantized values:", quantized_data)
print("Scale:", scale)
print("Zero-point:", zero_point)

Original values: [-1.0, -0.5, 0.0, 0.5, 1.0]
Quantized values: [-128  -65   -1   63  126]
Scale: 0.007843138
Zero-point: -1


Quantization converts float32 values into INT8 values, significantly reducing storage requirements.

Some numerical precision is lost because continuous floating-point values are represented using discrete integer values.

# Task 2: Estimate Model Size

Model size depends on the number of parameters and the number of bytes
required to store each parameter.

FP32 = 4 bytes
FP16 = 2 bytes
INT8 = 1 byte

Formula:

Model Size = Number of Parameters × Bytes per Parameter


In [24]:
def model_size(num_parameters, dtype):
    bytes_per_parameter = {
        "fp32": 4,
        "fp16": 2,
        "int8": 1
    }

    dtype = dtype.lower()

    if dtype not in bytes_per_parameter:
        raise ValueError("dtype must be fp32, fp16, or int8")

    size_bytes = num_parameters * bytes_per_parameter[dtype]

    size_mb = size_bytes / (1024 ** 2)
    size_gb = size_bytes / (1024 ** 3)

    return size_mb, size_gb

In [35]:
parameters = 125_000_000

for dtype in ["fp32", "fp16", "int8"]:
    mb, gb = model_size(parameters, dtype)

    print(dtype.upper(), ":", round(mb, 2), "MB",
          "|", round(gb, 4), "GB")

FP32 : 476.84 MB | 0.4657 GB
FP16 : 238.42 MB | 0.2328 GB
INT8 : 119.21 MB | 0.1164 GB


In [41]:
models = {
    "125M": 125_000_000,
    "1B": 1_000_000_000,
    "7B": 7_000_000_000
}

for model_name, parameters in models.items():

    print("\n", model_name)

    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(parameters, dtype)

        print(
            dtype.upper(),
            "->",
            round(mb, 2),
            "MB |",
            round(gb, 2),
            "GB"
        )


 125M
FP32 -> 476.84 MB | 0.47 GB
FP16 -> 238.42 MB | 0.23 GB
INT8 -> 119.21 MB | 0.12 GB

 1B
FP32 -> 3814.7 MB | 3.73 GB
FP16 -> 1907.35 MB | 1.86 GB
INT8 -> 953.67 MB | 0.93 GB

 7B
FP32 -> 26702.88 MB | 26.08 GB
FP16 -> 13351.44 MB | 13.04 GB
INT8 -> 6675.72 MB | 6.52 GB


### Observation

INT8 requires approximately one-fourth of the storage required by FP32.
Therefore, converting FP32 weights to INT8 can reduce raw parameter storage by approximately 75%.

## Multi-GPU Inference Design

The model should first be split across GPUs within the same node because
the 8 GPUs are connected through NVLink. NVLink provides fast and
low-latency GPU-to-GPU communication, which is useful when different
parts of a model frequently exchange intermediate activations.

If the model is still too large for the 8 GPUs in one node, model
parallelism can be extended across multiple nodes. Communication between
nodes should use InfiniBand.

NVLink is therefore mainly used for intra-node GPU communication, while
InfiniBand is used for inter-node communication. Swapping these roles
would increase communication overhead and could reduce inference
performance because communication would use a less suitable interconnect.

In [47]:
def determine_bound(
    flops_per_token,
    compute_throughput,
    memory_traffic,
    memory_bandwidth
):

    compute_time = flops_per_token / compute_throughput

    memory_time = memory_traffic / memory_bandwidth

    if compute_time > memory_time:
        bound = "Compute-bound"
    else:
        bound = "Memory-bound"

    return compute_time, memory_time, bound

In [54]:
configs = [
    {
        "name": "Compute Heavy",
        "flops": 100e12,
        "compute": 10e12,
        "memory": 500e9,
        "bandwidth": 2e12
    },
    {
        "name": "Memory Heavy",
        "flops": 10e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e9
    },
    {
        "name": "Balanced",
        "flops": 20e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e12
    }
]

In [64]:
configs = [
    {
        "name": "Compute Heavy",
        "flops": 100e12,
        "compute": 10e12,
        "memory": 100e9,
        "bandwidth": 2e12
    },
    {
        "name": "Memory Heavy",
        "flops": 10e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e9
    },
    {
        "name": "Balanced",
        "flops": 20e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e12
    }
]

In [31]:
for config in configs:

    compute_time, memory_time, bound = determine_bound(
        config["flops"],
        config["compute"],
        config["memory"],
        config["bandwidth"]
    )

    print("\n", config["name"])
    print("Compute Time:", compute_time, "seconds")
    print("Memory Time:", memory_time, "seconds")
    print("Result:", bound)


 Compute Heavy
Compute Time: 10.0 seconds
Memory Time: 0.05 seconds
Result: Compute-bound

 Memory Heavy
Compute Time: 1.0 seconds
Memory Time: 2.0 seconds
Result: Memory-bound

 Balanced
Compute Time: 2.0 seconds
Memory Time: 0.002 seconds
Result: Compute-bound


In [66]:
balanced = {
    "name": "Balanced",
    "flops": 20e12,
    "compute": 10e12,
    "memory": 1e12,
    "bandwidth": 500e9
}


In [69]:
def determine_bound(
    flops_per_token,
    compute_throughput,
    memory_traffic,
    memory_bandwidth
):

    compute_time = flops_per_token / compute_throughput
    memory_time = memory_traffic / memory_bandwidth

    if compute_time > memory_time:
        bound = "Compute-bound"
    else:
        bound = "Memory-bound"

    return compute_time, memory_time, bound

In [70]:
configs = [
    {
        "name": "Compute Heavy",
        "flops": 100e12,
        "compute": 10e12,
        "memory": 100e9,
        "bandwidth": 2e12
    },
    {
        "name": "Memory Heavy",
        "flops": 10e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e9
    },
    {
        "name": "Balanced",
        "flops": 20e12,
        "compute": 10e12,
        "memory": 1e12,
        "bandwidth": 500e9
    }
]

for config in configs:

    compute_time, memory_time, bound = determine_bound(
        config["flops"],
        config["compute"],
        config["memory"],
        config["bandwidth"]
    )

    print("\n" + config["name"])
    print("Compute Time:", round(compute_time, 4), "seconds")
    print("Memory Time:", round(memory_time, 4), "seconds")
    print("Result:", bound)


Compute Heavy
Compute Time: 10.0 seconds
Memory Time: 0.05 seconds
Result: Compute-bound

Memory Heavy
Compute Time: 1.0 seconds
Memory Time: 2.0 seconds
Result: Memory-bound

Balanced
Compute Time: 2.0 seconds
Memory Time: 2.0 seconds
Result: Memory-bound


### Observation

For compute-bound inference, reducing computation, optimizing kernels, or
using lower precision can provide greater benefits. For memory-bound
inference, reducing memory traffic and using quantization can provide
greater benefits.